**PRÁCTICA 4: DETECCIÓN Y LECTURA DE MATRÍCULAS CON YOLO Y PADDLEOCR**

En este cuaderno se desarrolla el sistema de detección, lectura y seguimiento de matrículas mediante modelos de visión por computador y reconocimiento óptico de caracteres (OCR).

El flujo de trabajo se estructura en:
1. **Configuración inicial e imports necesarios**: Carga de librerías comunes y definición global de los modelos de detección y OCR.
2. **Utilidades comunes y función unificada de lectura**: Operaciones geométricas, métricas de similitud de cadenas y la función única `leer_placa`.
3. **Adquisición de métricas de OCR**: Evaluación cuantitativa del sistema sobre el conjunto de datos de placas (extremo a extremo vs recorte de anotación).
4. **Lectura y seguimiento en vídeo**: Tracking de vehículos y peatones mediante ByteTrack, asociación y filtrado temporal de placas, y renderizado del vídeo final.


**IMPORTS NECESARIOS Y CONFIGURACIÓN GLOBAL**

Para llevar a cabo la detección y lectura de matrículas, inicialmente se importan las librerías necesarias y se configuran las dependencias del sistema.


In [2]:
import os
import re
import csv
import json
import math
from pathlib import Path
from collections import defaultdict, deque
from time import perf_counter

import cv2
import numpy as np
from paddleocr import PaddleOCR
from ultralytics import YOLO


**INICIALIZACIÓN GLOBAL DE MODELOS**

Definimos una única instancia global para el detector de matrículas entrenado (`best-yolo.pt`) y para el motor de reconocimiento PaddleOCR. De esta forma evitamos recargar modelos en memoria o duplicar inicializaciones entre la adquisición de métricas y el procesado de vídeo.


In [ ]:
# Ruta al detector entrenado de placas
WEIGHTS_PLACAS = "weights/best-yolo.pt"

# Inicialización del detector de matrículas compartido
detector_placas = YOLO(WEIGHTS_PLACAS)

# Instanciación y gestión del motor OCR global
ocr_global = None
_ocr_disponible = True

def get_ocr():
    """Devuelve la instancia global de PaddleOCR, inicializándola solo la primera vez."""
    global ocr_global, _ocr_disponible
    if not _ocr_disponible:
        return None
    if ocr_global is None:
        try:
            ocr_global = PaddleOCR(
                lang='es',
                use_textline_orientation=True,       
                use_doc_orientation_classify=False,  
                use_doc_unwarping=False,             
                device='cpu',
                enable_mkldnn=False,
            )
        except ImportError:
            print("AVISO: paddleocr no está instalado (pip install paddlepaddle paddleocr)")
            _ocr_disponible = False
            return None
    return ocr_global

# Inicializamos el OCR globalmente
ocr_global = get_ocr()


**UTILIDADES COMUNES: GEOMETRÍA, CADENAS Y MANIPULACIÓN DE CAJAS**

A continuación se definen las funciones auxiliares compartidas necesarias para el tratamiento de texto, cálculo de solapamiento de cajas (IoU), cálculo de centros y distancias, y recorte de regiones de interés.


In [4]:
# Normaliza texto a mayúsculas alfanuméricas
def norm(s: str) -> str:
    return re.sub(r"[^A-Z0-9]", "", s.upper())


# Distancia de Levenshtein entre dos cadenas de caracteres
def lev(a: str, b: str) -> int:
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]


# Devuelve el cálculo del solapamiento (IoU) entre dos regiones pasadas
def iou(a, b):
    x1, y1 = max(a[0], b[0]), max(a[1], b[1])
    x2, y2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    ua = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / ua if ua > 0 else 0.0


# Centroide de una caja [x1, y1, x2, y2]
def centro(b):
    return ((b[0] + b[2]) / 2, (b[1] + b[3]) / 2)


# Diagonal de una caja para cálculo de tamaño relativo
def diag(b):
    return math.hypot(b[2] - b[0], b[3] - b[1])


# Recorta una región con margen porcentual o en píxeles ajustada a los límites de la imagen
def recortar(img, box, pad=0.05, margen_px=None):
    h, w = img.shape[:2]
    x1, y1, x2, y2 = box
    if margen_px is not None:
        return img[max(0, y1 - margen_px):min(h, y2 + margen_px),
                   max(0, x1 - margen_px):min(w, x2 + margen_px)]
    bw, bh = x2 - x1, y2 - y1
    x1 = max(0, int(x1 - pad * bw))
    x2 = min(w, int(x2 + pad * bw))
    y1 = max(0, int(y1 - pad * bh))
    y2 = min(h, int(y2 + pad * bh))
    return img[y1:y2, x1:x2]


**FUNCIÓN ÚNICA DE DETECCIÓN Y LECTURA DE MATRÍCULAS (`leer_placa`)**

Implementa la lectura OCR de la matrícula a partir del recorte de imagen. Combina un preprocesamiento que asegura dimensiones mínimas mediante interpolación bicúbica, filtrado Gaussiano y padding réplica, para posteriormente ordenar espacialmente el texto detectado (soportando placas de una o dos líneas) y calcular la confianza media de lectura.


In [ ]:
def leer_placa(crop, ocr_inst=None):
    """Devuelve (texto normalizado, confianza media del OCR) usando PaddleOCR."""
    if crop is None or crop.size == 0:
        return "", 0.0

    reader = ocr_inst if ocr_inst is not None else get_ocr()
    if reader is None:
        return "", 0.0

    # Escala mínima garantizada tanto en alto como en ancho
    h_c, w_c = crop.shape[:2]
    esc = max(max(1.0, 64.0 / h_c), max(1.0, 200.0 / w_c))
    
    g = cv2.resize(crop, None, fx=esc, fy=esc, interpolation=cv2.INTER_CUBIC)
    g = cv2.GaussianBlur(g, (3, 3), 0)
    g = cv2.copyMakeBorder(g, 10, 10, 10, 10, cv2.BORDER_REPLICATE)

    out = []
    for page in reader.predict(g) or []:
        rec_texts = page.get("rec_texts", [])
        rec_scores = page.get("rec_scores", [])
        rec_polys = page.get("rec_polys", [])
        
        if rec_polys is not None and len(rec_polys) == len(rec_texts):
            for t, s, p in zip(rec_texts, rec_scores, rec_polys):
                p = np.asarray(p)
                out.append((t, float(s), float(p[:, 0].mean()), float(p[:, 1].mean())))
        else:
            for t, s in zip(rec_texts, rec_scores):
                out.append((t, float(s), 0.0, 0.0))

    out = [r for r in out if norm(r[0])]
    if not out:
        return "", 0.0

    # Ordenar por línea (vertical) y luego de izquierda a derecha
    alto = g.shape[0]
    out.sort(key=lambda r: (round(r[3] / (alto / 2)), r[2]))

    texto = norm("".join(r[0] for r in out))
    confianza = float(np.mean([r[1] for r in out]))
    return texto, confianza


**TAREA 1: ADQUISICIÓN Y EVALUACIÓN DE MÉTRICAS OCR**

En esta fase se evalúa el rendimiento cuantitativo del sistema sobre el conjunto de anotaciones reales (Ground Truth):
- **Exact Match**: Porcentaje de matrículas donde la cadena leída coincide al 100% con el texto real.
- **Acierto por carácter**: Precisión media basada en la distancia normalizada de Levenshtein: $1 - \min(1, \text{lev} / \text{longitud})$.
- **Tiempos de ejecución (`perf_counter`)**: Se mide el tiempo medio empleado por detección tanto en localización (YOLO) como en reconocimiento (PaddleOCR), así como el coste temporal total del pipeline por detección.
- **Análisis de falsos positivos**: Se diferencian las detecciones sin coincidencia con ninguna caja real (falsos positivos geométricos) de aquellas que coinciden con una placa anotada pero sin texto legible (`None`/vacía).


In [ ]:
# Configuración de rutas y parámetros del dataset
BASE = Path("plates/raw-dataset")
IMAGENES = BASE / "images"
ANOTACIONES = BASE / "annotations"
SPLIT = "train"
MAX_IMAGENES = 200     # None para evaluar el dataset completo
GT_KEYS = ("lpn", "text", "plate", "label", "number", "plate_number")
CONF_METRICAS = 0.25
IOU_MATCH = 0.5
PAD = 0.05
IMGSZ_METRICAS = 960
EXTS = {".jpg", ".jpeg", ".png"}

SALIDA = Path("ocr_eval")
SALIDA.mkdir(exist_ok=True)
(SALIDA / "errores-yolo").mkdir(exist_ok=True)


# Carga las cajas y textos anotados desde los archivos JSON
def cargar_gt(img_path, w, h):
    ruta = ANOTACIONES / f"{img_path.stem}.json"
    if not ruta.exists():
        return []
    gts = []
    for obj in json.loads(ruta.read_text()):
        xc, yc, bw, bh = obj["rectangle"]
        box = ((xc - bw / 2) * w, (yc - bh / 2) * h, (xc + bw / 2) * w, (yc + bh / 2) * h)
        txt = next((norm(str(obj[k])) for k in GT_KEYS if obj.get(k)), None)
        gts.append((box, txt))
    return gts


# Muestra el resumen de exact match y acierto medio por carácter
def resumen_metricas(nombre, rows):
    rows = [r for r in rows if r["gt"]]
    if not rows:
        return
    exact = np.mean([r["gt"] == r["ocr"] for r in rows])
    acc_car = np.mean([1 - min(1, lev(r["gt"], r["ocr"]) / max(len(r["gt"]), 1)) for r in rows])
    print(f"{nombre:<34} n={len(rows):<5} exact match={exact:.3f} | acierto por carácter={acc_car:.3f}")


A continuación se implementa la función `evaluar_dataset`. Emplea `perf_counter` para registrar con alta precisión los tiempos medios de lectura OCR por detección (tanto a partir de las detecciones de YOLO como a partir de los recortes anotados del Ground Truth), clasifica rigurosamente los posibles falsos positivos geométricos frente a placas anotadas con texto `None`, y genera tanto el archivo `ocr_resultados.csv` como el guardado de recortes erróneos en disco.


In [ ]:
def evaluar_dataset(lista_imagenes, detector=detector_placas):
    """Evalúa OCR en detecciones YOLO y en recortes anotados (techo OCR), midiendo tiempos por detección con perf_counter."""
    filas = []
    n_gt = n_gt_texto = n_detectadas = 0
    tiempos_ocr_det = []
    tiempos_ocr_gt = []
    falsos_positivos_geom = 0
    detecciones_en_gt_sin_texto = 0

    for img_path in lista_imagenes:
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        h, w = img.shape[:2]
        gts = cargar_gt(img_path, w, h)
        n_gt += len(gts)
        n_gt_texto += sum(t is not None for _, t in gts)

        # Inferencia de detección de placas
        r = detector.predict(img, imgsz=IMGSZ_METRICAS, conf=CONF_METRICAS, verbose=False)[0]

        preds = [(b.xyxy[0].tolist(), float(b.conf[0])) for b in r.boxes]
        n_detectadas += len(preds)

        # OCR a partir de las detecciones de YOLO
        for box, det_conf in preds:
            t_ini_det = perf_counter()
            rec = recortar(img, box, pad=PAD)
            texto, ocr_conf = leer_placa(rec)
            tiempos_ocr_det.append(perf_counter() - t_ini_det)

            mejor_iou, gt_txt = 0.0, None
            for gbox, gtxt in gts:
                v = iou(box, gbox)
                if v > mejor_iou:
                    mejor_iou, gt_txt = v, gtxt

            # Análisis de emparejamiento con el Ground Truth
            if mejor_iou < IOU_MATCH:
                falsos_positivos_geom += 1
                continue
            elif gt_txt is None or gt_txt == "NONE":
                detecciones_en_gt_sin_texto += 1
                continue

            filas.append(dict(imagen=img_path.name, origen="deteccion", gt=gt_txt, ocr=texto,
                              det_conf=round(det_conf, 3), ocr_conf=round(ocr_conf, 3),
                              iou=round(mejor_iou, 3)))
            if gt_txt != texto:
                cv2.imwrite(str(SALIDA / "errores-yolo" / f"{len(filas):05d}_gt-{gt_txt}_ocr-{texto}.jpg"), rec)

        # 2) OCR a partir del recorte de la anotación = techo del OCR
        for gbox, gtxt in gts:
            if gtxt is None or gtxt == "NONE":
                continue
            t_ini_gt = perf_counter()
            rec_gt = recortar(img, gbox, pad=PAD)
            texto, ocr_conf = leer_placa(rec_gt)
            tiempos_ocr_gt.append(perf_counter() - t_ini_gt)

            filas.append(dict(imagen=img_path.name, origen="gt_recorte", gt=gtxt, ocr=texto,
                              det_conf=None, ocr_conf=round(ocr_conf, 3), iou=1.0))

    # Guardar resultados en CSV
    csv_path = SALIDA / "yolo_resultados_ocr.csv"
    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        wr = csv.DictWriter(f, fieldnames=list(filas[0].keys()) if filas else ["vacio"], delimiter=";")
        wr.writeheader()
        wr.writerows(filas)

    # Métricas de tiempo medio de ejecución por detección (extra)
    tiempo_medio_ocr_det = (np.mean(tiempos_ocr_det) * 1000) if tiempos_ocr_det else 0.0
    tiempo_medio_ocr_gt = (np.mean(tiempos_ocr_gt) * 1000) if tiempos_ocr_gt else 0.0

    print(f"\nImágenes evaluadas: {len(lista_imagenes)} | placas anotadas: {n_gt} (con texto: {n_gt_texto}) | detecciones YOLO: {n_detectadas}")
    print(f"--- TIEMPOS MEDIOS DE LECTURA POR DETECCIÓN (perf_counter) ---")
    print(f"OCR a partir de detección YOLO:     {tiempo_medio_ocr_det:6.2f} ms / detección")
    print(f"OCR a partir de recorte GT (techo): {tiempo_medio_ocr_gt:6.2f} ms / recorte anotado")

    if n_gt_texto == 0:
        print(f"\nLas anotaciones no contienen texto de matrícula (claves probadas: {GT_KEYS}).")
    else:
        print(f"\n--- RESUMEN DE EXACTITUD OCR ---")
        resumen_metricas("OCR sobre detecciones de YOLO", [r for r in filas if r["origen"] == "deteccion"])
        resumen_metricas("OCR sobre recorte anotado (techo)", [r for r in filas if r["origen"] == "gt_recorte"])
        emp = sum(1 for r in filas if r["origen"] == "deteccion" and r["gt"])
        print(f"Placas evaluadas encontradas por YOLO: {emp}/{n_gt_texto}")
        print(f"Falsos positivos geométricos (IoU < {IOU_MATCH}): {falsos_positivos_geom}")
        print(f"Detecciones en placas anotadas sin texto (None): {detecciones_en_gt_sin_texto}")

    print(f"\nCSV: {csv_path}\nRecortes con error: {SALIDA / 'errores'}")
    return filas


# Ejecución de la evaluación de métricas
if IMAGENES.exists():
    lista = sorted(p for p in IMAGENES.iterdir() if p.suffix.lower() in EXTS)
    if MAX_IMAGENES:
        lista = lista[:MAX_IMAGENES]
    evaluar_dataset(lista)
else:
    print(f"Directorio {IMAGENES} no encontrado. Comprueba la ruta del dataset.")



Imágenes evaluadas: 200 | placas anotadas: 407 (con texto: 407) | detecciones YOLO: 367
--- TIEMPOS MEDIOS DE LECTURA POR DETECCIÓN (perf_counter) ---
OCR a partir de detección YOLO:     744.38 ms / detección
OCR a partir de recorte GT (techo): 933.89 ms / recorte anotado

--- RESUMEN DE EXACTITUD OCR ---
OCR sobre detecciones de YOLO      n=188   exact match=0.734 | acierto por carácter=0.894
OCR sobre recorte anotado (techo)  n=210   exact match=0.648 | acierto por carácter=0.791
Placas evaluadas encontradas por YOLO: 188/407
Falsos positivos geométricos (IoU < 0.5): 108
Detecciones en placas anotadas sin texto (None): 71

CSV: ocr_eval\ocr_resultados.csv
Recortes con error: ocr_eval\errores


**TAREA 2: DETECCIÓN, TRACKING Y LECTURA DE MATRÍCULAS EN VÍDEO**

En este módulo se procesa una secuencia de vídeo realizando:
- **Seguimiento con ByteTrack**: Detección y seguimiento continuo de personas y vehículos (`persona`, `coche`, `moto`, `guagua`, `camion`).
- **Filtrado y validación sintáctica**: Verificación de las matrículas leídas mediante expresiones regulares de formatos de placas europeos.
- **Fusión de tracklets**: Recuperación de identificadores ante oclusiones temporales, tanto para vehículos en movimiento como para vehículos estacionados.
- **Renderizado y exportación**: Generación del vídeo anotado con los identificadores y recuadros de matrícula, junto con el archivo `resultado.csv` detallando las detecciones por fotograma.


In [8]:
# Configuración de rutas de vídeo
filename = "C0142.MP4"
carpeta = os.path.join(os.path.dirname(os.path.abspath(filename)), "YOLO_PaddleOCR_result")
os.makedirs(carpeta, exist_ok=True)
output_path = os.path.join(carpeta, "resultado.mp4")
csv_path = os.path.join(carpeta, "resultado.csv")

# Mapeo de clases y paleta visual
nombre = {0: "persona", 2: "coche", 3: "moto", 5: "guagua", 7: "camion"}
colores = {0: (0, 255, 0), 2: (0, 0, 255), 3: (255, 0, 255),
           5: (0, 255, 255), 7: (255, 255, 0)}

COLOR_PLACA = (0, 165, 255)
ESCALA_TEXTO_PLACA = 0.35
CONF_PLACA = 0.4
MIN_FRAMES_PLACA = 5

# Parámetros del OCR sobre vídeo
OCR_CADA = 5
MIN_ANCHO_OCR = 40
MIN_CHARS_PLACA = 4
OCR_SCORE_OK = 0.5
OCR_MAX_INTENTOS = 6

# Patrones sintácticos de matrículas europeas
CONS = "BCDFGHJKLMNPRSTVWXYZ"
PATRONES_MATRICULA = [re.compile(p) for p in (
    rf"\d{{4}}[{CONS}]{{3}}",              # España actual
    r"[A-Z]{1,2}\d{4}[A-Z]{1,2}",          # España antigua
    r"[A-Z]{2}\d{3}[A-Z]{2}",              # Francia / Italia
    r"[A-Z]{2}\d{2}[A-Z]{3}",              # Reino Unido
    r"[A-Z]{1,3}[A-Z]{1,2}\d{1,4}[EH]?",   # Alemania
    r"\d{2}[A-Z]{2}\d{2}",                 # Portugal
    r"[A-Z]{2}\d{2}[A-Z]{2}",              # Portugal
    r"\d{2}\d{2}[A-Z]{2}",                 # Portugal
    r"[1-9][A-Z]{3}\d{3}",                 # Bélgica
    r"\d{2}[A-Z]{3}\d",                    # Países Bajos
    r"\d[A-Z]{3}\d{2}",
    r"[A-Z]{2}\d{3}[A-Z]",
    r"\d{3}[A-Z]{2}\d",
    r"[A-Z]\d{3}[A-Z]{2}",
    r"[A-Z]{3}\d{2}[A-Z]",
    r"[A-Z]{3}\d{3}",                      # Polonia / otros
    r"[A-Z]{2}\d{5}",
)]

# Parámetros de tracking y fusión de tracklets
FAMILIA = {0: 'p', 3: 'm', 2: 'v', 5: 'v', 7: 'v'}
PARAM_FUSION = {'p': (5, 0.5, 0.01), 'v': (5, 0.5, 0.01), 'm': (10, 0.8, 0.02)}
GAP_ESTATICO_SEG = 120
MIN_FRAMES_ESTATICO = 15
IOU_ESTATICO = 0.5
MIN_FRAMES = 10
MAYORIA_PESADOS = 0.65
PESO_BORDE = 0.3
MARGEN_BORDE = 5

# Configuración del archivo de ByteTrack
os.makedirs("./YOLO", exist_ok=True)
with open("./YOLO/mi_bytetrack.yaml", "w") as f:
    f.write("tracker_type: bytetrack\ntrack_high_thresh: 0.25\ntrack_low_thresh: 0.1\n"
            "new_track_thresh: 0.25\ntrack_buffer: 140\nmatch_thresh: 0.8\nfuse_score: True\n")


A continuación se implementan las funciones encargadas de:
- Validar el formato de la matrícula según los patrones definidos (`matricula_valida`).
- Extrapolar la posición temporal del objeto ante pérdida de fotogramas (`prediccion_posicion`).
- Distinguir vehículos aparcados de objetos en movimiento (`es_estatico`).
- Realizar la fusión de tracklets fragmentados por oclusión (`fusionar_tracklets`).
- Resolver la clase final del objeto por votación mayoritaria (`decidir_clase`).


In [9]:
# Valida si el texto coincide con alguna sintaxis europea conocida
def matricula_valida(texto):
    return any(p.fullmatch(texto) for p in PATRONES_MATRICULA)


# Estima la posición esperada del tracklet tras 'gap' fotogramas perdidos
def prediccion_posicion(info, a, gap):
    (f0, c0), (f1, c1) = info[a]['hist'][0], info[a]['hist'][-1]
    if f1 == f0:
        return c1
    vx, vy = (c1[0] - c0[0]) / (f1 - f0), (c1[1] - c0[1]) / (f1 - f0)
    g = min(gap, 60)
    return (c1[0] + vx * g, c1[1] + vy * g)


# Comprueba si un objeto ha permanecido inmóvil
def es_estatico(info, frames_visto, t):
    if frames_visto[t] < MIN_FRAMES_ESTATICO:
        return False
    mov = math.dist(centro(info[t]['caja_ini']), centro(info[t]['caja_fin']))
    return mov / max(diag(info[t]['caja_fin']), 1) < 0.15


# Algoritmo de unión de tracklets por doble vía (vehículos aparcados o trayectoria)
def fusionar_tracklets(info, frames_visto, clase_prelim, fps):
    ids = sorted(info, key=lambda t: info[t]['ini'])
    raiz = {t: t for t in ids}
    con_sucesor = set()
    for b in ids:
        fam = FAMILIA[clase_prelim[b]]
        seg, base, por_frame = PARAM_FUSION[fam]
        b_estatico_ok = frames_visto[b] < MIN_FRAMES_ESTATICO or es_estatico(info, frames_visto, b)
        mejor, mejor_d = None, 1e9
        for a in ids:
            if a in con_sucesor or info[a]['fin'] >= info[b]['ini']:
                continue
            if FAMILIA[clase_prelim[a]] != fam:
                continue
            gap = info[b]['ini'] - info[a]['fin']
            da, db = diag(info[a]['caja_fin']), diag(info[b]['caja_ini'])

            # Vía estática: vehículo aparcado con oclusión prolongada
            if (fam != 'p' and gap <= GAP_ESTATICO_SEG * fps and b_estatico_ok
                    and es_estatico(info, frames_visto, a) and 0.7 <= db / max(da, 1) <= 1.4):
                s = iou(info[a]['caja_fin'], info[b]['caja_ini'])
                if s >= IOU_ESTATICO:
                    d = 0.01 * (1 - s)
                    if d < mejor_d:
                        mejor, mejor_d = a, d
                    continue

            # Vía dinámica: movimiento continuo y cercanía a la posición predicha
            if gap > seg * fps:
                continue
            if not 0.4 <= db / max(da, 1) <= 2.5:
                continue
            cb = centro(info[b]['caja_ini'])
            d = min(math.dist(centro(info[a]['caja_fin']), cb),
                    math.dist(prediccion_posicion(info, a, gap), cb)) / max(da, 1)
            if d <= base + por_frame * gap and d < mejor_d:
                mejor, mejor_d = a, d
        if mejor is not None:
            raiz[b] = raiz[mejor]
            con_sucesor.add(mejor)
    return raiz


# Decide la clase del objeto por mayoría ponderada
def decidir_clase(v):
    c = max(v, key=v.get)
    if c in (5, 7) and v[c] / sum(v.values()) < MAYORIA_PESADOS:
        return 2
    return c


Finalmente, se define la función `procesar_video` que integra las dos pasadas:
1. **Pasada 1**: Ejecución de inferencia sobre el vídeo, seguimiento con ByteTrack, detección de matrículas mediante `detector_placas`, lectura mediante `leer_placa` y fusión temporal de identidades.
2. **Pasada 2**: Renderizado del vídeo final (`resultado.mp4`) superponiendo las cajas y matrículas leídas, y volcado a `resultado.csv`.


In [ ]:
def procesar_video(video_path, salida_video, salida_csv, detector_placas_inst=detector_placas):
    """Procesamiento en dos pasadas: inferencia/tracking y renderizado/exportación CSV."""
    model_yolo = YOLO('yolo11l.pt')

    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()

    # ---------- Pasada 1: Detección, tracking y lectura OCR ----------
    print("Iniciando Pasada 1: tracking, detección y lectura OCR en vídeo...")
    results = model_yolo.track(source=video_path, stream=True, classes=list(nombre),
                               conf=0.5, iou=0.5, imgsz=960, persist=True,
                               tracker="./YOLO/mi_bytetrack.yaml")

    datos = []
    placas_datos = []
    votos = defaultdict(lambda: defaultdict(float))
    frames_visto = defaultdict(int)
    info = {}
    lecturas = {}
    ultimo_ocr = {}
    intentos_ocr = defaultdict(int)

    for i, r in enumerate(results):
        frame_data = []
        if r.boxes.id is not None:
            for caja, tid, c, conf in zip(r.boxes.xyxy.int().tolist(), r.boxes.id.int().tolist(),
                                          r.boxes.cls.int().tolist(), r.boxes.conf.tolist()):
                x1, y1, x2, y2 = caja
                borde = (x1 <= MARGEN_BORDE or y1 <= MARGEN_BORDE or
                         x2 >= w - MARGEN_BORDE or y2 >= h - MARGEN_BORDE)
                votos[tid][c] += conf * (PESO_BORDE if borde else 1.0)
                frames_visto[tid] += 1
                info.setdefault(tid, {'ini': i, 'caja_ini': caja, 'hist': deque(maxlen=15)})
                info[tid]['fin'], info[tid]['caja_fin'] = i, caja
                info[tid]['hist'].append((i, ((x1 + x2) / 2, (y1 + y2) / 2)))
                frame_data.append((caja, tid, c, conf))
        datos.append(frame_data)

        # Matrículas sobre este mismo fotograma usando detector_placas_inst
        placas_f = []
        if any(c != 0 for _, _, c, _ in frame_data):
            rp = detector_placas_inst.predict(r.orig_img, conf=CONF_PLACA, imgsz=960, verbose=False)[0]
            for pbox, pconf in zip(rp.boxes.xyxy.int().tolist(), rp.boxes.conf.tolist()):
                cx, cy = (pbox[0] + pbox[2]) / 2, (pbox[1] + pbox[3]) / 2
                cands = sorted(((b[2] - b[0]) * (b[3] - b[1]), t) for b, t, c, _ in frame_data
                               if c != 0 and b[0] <= cx <= b[2] and b[1] <= cy <= b[3])
                if not cands:
                    continue
                cands = [t for _, t in cands]
                t0 = cands[0]
                if (pbox[2] - pbox[0] >= MIN_ANCHO_OCR
                        and intentos_ocr[t0] < OCR_MAX_INTENTOS
                        and lecturas.get(t0, (0.0, ""))[0] < OCR_SCORE_OK
                        and (t0 not in ultimo_ocr or i - ultimo_ocr[t0] >= OCR_CADA)):
                    ultimo_ocr[t0] = i
                    intentos_ocr[t0] += 1
                    rec_placa = recortar(r.orig_img, pbox, margen_px=4)
                    texto, oconf = leer_placa(rec_placa)
                    if len(texto) >= MIN_CHARS_PLACA and matricula_valida(texto):
                        score = pconf * oconf
                        if t0 not in lecturas or score > lecturas[t0][0]:
                            lecturas[t0] = (score, texto)
                placas_f.append((pbox, pconf, cands))
        placas_datos.append(placas_f)

    # Fusión de tracklets y asignación de lecturas OCR por vehículo
    raiz = fusionar_tracklets(info, frames_visto, {t: max(v, key=v.get) for t, v in votos.items()}, fps)

    votos_m = defaultdict(lambda: defaultdict(float))
    frames_m = defaultdict(int)
    for tid, v in votos.items():
        for c, x in v.items():
            votos_m[raiz[tid]][c] += x
        frames_m[raiz[tid]] += frames_visto[tid]

    clase_final = {tid: decidir_clase(v) for tid, v in votos_m.items()}

    texto_veh = {}
    for t, (score, texto) in lecturas.items():
        v = raiz[t]
        if v not in texto_veh or score > texto_veh[v][0]:
            texto_veh[v] = (score, texto)
    texto_veh = {v: texto for v, (_, texto) in texto_veh.items()}

    print(f"IDs originales: {len(info)} -> tras fusionar: {len(set(raiz.values()))}")

    # ---------- Pasada 2: Renderizado de vídeo y guardado a CSV ----------
    print("Iniciando Pasada 2: renderizado de vídeo y escritura en CSV...")
    ids_propios = {c: {} for c in nombre}
    placas_vehiculo = set()
    frames_placa = defaultdict(int)

    cap = cv2.VideoCapture(video_path)
    writer = cv2.VideoWriter(salida_video, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))

    csv_file = open(salida_csv, "w", newline="", encoding="utf-8")
    csv_out = csv.writer(csv_file, delimiter=';')
    csv_out.writerow(["fotograma", "tipo_objeto", "confianza_objeto", "identificador_tracking",
                      "x1", "y1", "x2", "y2",
                      "matricula", "confianza_matricula", "mx1", "my1", "mx2", "my2",
                      "texto_matricula"])

    for i, (frame_data, placas) in enumerate(zip(datos, placas_datos)):
        ok, frame = cap.read()
        if not ok:
            break

        visibles = {}
        for caja, tid, _, conf in frame_data:
            vid = raiz[tid]
            if frames_m[vid] < MIN_FRAMES:
                continue
            c = clase_final[vid]
            ids_propios[c].setdefault(vid, len(ids_propios[c]) + 1)
            visibles[vid] = (caja, c, conf)

        placa_de = {}
        for pbox, pconf, cands in placas:
            vid = next((raiz[t] for t in cands
                        if raiz[t] in visibles and visibles[raiz[t]][1] != 0), None)
            if vid is None:
                continue
            frames_placa[vid] += 1
            if frames_placa[vid] < MIN_FRAMES_PLACA:
                continue
            placas_vehiculo.add(vid)
            if vid not in placa_de or pconf > placa_de[vid][1]:
                placa_de[vid] = (pbox, pconf)

        for vid, (caja, c, conf) in visibles.items():
            x1, y1, x2, y2 = caja
            idp = ids_propios[c][vid]
            etiqueta = f"{nombre[c]} {idp}"
            tiene_texto = vid in texto_veh
            cv2.rectangle(frame, (x1, y1), (x2, y2), colores[c], 2)
            cv2.putText(frame, etiqueta, (x1, max(y1 - 8, 15)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, colores[c], 2)

            if vid in placa_de:
                (px1, py1, px2, py2), pconf = placa_de[vid]
                cv2.rectangle(frame, (px1, py1), (px2, py2), COLOR_PLACA, 2)
                txt = texto_veh[vid] if tiene_texto else "?"
                fuente, esc, grosor = cv2.FONT_HERSHEY_SIMPLEX, ESCALA_TEXTO_PLACA, 1
                (tw, th), base = cv2.getTextSize(txt, fuente, esc, grosor)
                bx1 = max(0, px1)
                by2 = max(th + base + 6, py1 - 2)
                by1 = by2 - th - base - 6
                cv2.rectangle(frame, (bx1, by1), (bx1 + tw + 8, by2), COLOR_PLACA, -1)
                cv2.putText(frame, txt, (bx1 + 4, by2 - base - 3),
                            fuente, esc, (0, 0, 0), grosor)
                fila_placa = [1, f"{pconf:.3f}", px1, py1, px2, py2, texto_veh.get(vid, "")]
            else:
                fila_placa = [0, "", "", "", "", "", ""]
            csv_out.writerow([i, nombre[c], f"{conf:.3f}", idp, x1, y1, x2, y2] + fila_placa)

        cv2.rectangle(frame, (0, h - 40), (w, h), (0, 0, 0), -1)
        texto = "  |  ".join(f"{nombre[c]}: {len(ids_propios[c])}" for c in nombre)
        texto += f"  |  matricula: {len(placas_vehiculo)}"
        texto += f"  |  ocr: {len(texto_veh)}"
        cv2.putText(frame, texto, (10, h - 12), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        writer.write(frame)

    cap.release()
    writer.release()
    csv_file.close()

    print(f"Video guardado en {os.path.abspath(salida_video)} | CSV guardado en {os.path.abspath(salida_csv)}")
    print(f"Matrículas detectadas: {len(placas_vehiculo)} | Matrículas leídas con OCR: {len(texto_veh)}")


# Ejecución del pipeline de vídeo
if os.path.exists(filename):
    procesar_video(filename, output_path, csv_path)
else:
    print(f"Vídeo {filename} no encontrado. Comprueba la ruta.")
